# CodeAlpha Internship Task 2: Car Price Prediction with Machine Learning
**Author**: Murali 
**Objective**: Predict used car selling prices using regression algorithms (Linear Regression, Random Forest, Gradient Boosting), evaluate $R^2$, MAE, RMSE metrics, save the best model binary (`car_price_model.pkl`), and visualize feature importances.

## 1. Import Libraries & Setup Environment

In [ ]:
import os
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Dataset & Feature Engineering (`Car_Age` & One-Hot Encoding)

In [ ]:
df = pd.read_csv('car data.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Calculate Car_Age
current_year = datetime.datetime.now().year
df['car_age'] = current_year - df['year']

# Drop car_name and original year column
feature_df = df.drop(columns=['car_name', 'year'])

# One-Hot Encoding categorical features
encoded_df = pd.get_dummies(feature_df, columns=['fuel_type', 'selling_type', 'transmission'], drop_first=True)

X = encoded_df.drop(columns=['selling_price'])
y = encoded_df['selling_price']

display(encoded_df.head())

## 3. Train / Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

best_model_name = None
best_r2 = -float('inf')
best_model = None
best_y_pred = None

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    print(f"--- {name} ---")
    print(f"R² Score: {r2:.4f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}
")
    
    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name
        best_model = model
        best_y_pred = preds

# Save model binary
joblib.dump(best_model, os.path.join(OUTPUT_DIR, 'car_price_model.pkl'))
print(f"Saved Best Model ({best_model_name}) binary to outputs/car_price_model.pkl")

## 4. Visualizations: Feature Importance & Predictions

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    plt.figure(figsize=(10, 6))
    pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values().plot(kind='barh', color='#0275d8')
    plt.title(f'Feature Importances ({best_model_name})')
    plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=300)
    plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_y_pred, alpha=0.8, color='#d9534f', edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.title('Actual vs Predicted Car Selling Price')
plt.xlabel('Actual Price (Lakhs)')
plt.ylabel('Predicted Price (Lakhs)')
plt.savefig(os.path.join(OUTPUT_DIR, 'actual_vs_predicted_car_prices.png'), dpi=300)
plt.show()